[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/stage2_dash_skip_trigram_unigram_experiment.ipynb)

In [ ]:
# Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens tokenizers huggingface_hub plotly
    # transformer_lens pulls a newer torch; Colab's pre-installed torchaudio was built
    # against the old torch and its .so then fails to load (undefined symbol:
    # torch_library_impl). transformers imports torchaudio lazily and crashes on it,
    # though nothing here uses audio. Remove it so that import path is skipped.
    !pip uninstall -y -q torchaudio
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

# المرحلة ٢داش — تجربة Unigram: هل تجزئة Unigram بتخلّي الـ skip-trigram العربي أوضح من BPE؟

**النوتبوك ده للتعلّم بالتجربة.** نفس هيكل تجربة BPE بالظبط — الخلايا الأساسية (التحميل، التفكيك، bigram، تشخيص الرؤوس، الخريطة) شغّالة، وخلايا التحليل (§٥ب و§٥ج) فيها `# TODO(you)` دورك تكمّلها. الفرق الوحيد: الموديل ده مدرّب على **نفس الكوربَس بالظبط** (نفس الحروف، نفس البذرة ٤٢، نفس الإعدادات، vocab=١٢٠٠٠) بس بمُجزّئ **Unigram** بدل BPE — مقارنة محكومة بمتغيّر واحد.

**الفرضية (مقارنة):** هل تجزئة Unigram بتخلّي الـ skip-trigram العربي — خصوصًا أداة التعريف والتباين فصحى↔مصري — أوضح/أكثر محاذاة صرفيًا من BPE، اللي بيكسّر بعض البذور؟

**تنبيه أمين — الخسارتان مش قابلتين للمقارنة المباشرة:**
- Unigram: ٤٠٦.٥ مليون توكن، loss_end ٣.٩٥، ٧٣.٥ دقيقة.
- BPE: ٣٣٧.٦ مليون توكن، loss_end ٤.٩٦، ٦١.٩ دقيقة.
- Unigram بيكسّر **نفس النص** لعدد توكنز أكبر بـ~٢٠٪، فالتوزيع لكل توكن مختلف؛ **الخسارة الأقل لـ Unigram مش معناها موديل أحسن.** قارن الـ skip-trigrams اللي بتتأكّد، مش أرقام الخسارة.

**حدود مقدّمًا:** نفس حدود تجربة BPE — التشخيص على **جملة واحدة**، **عيب بنيوي** في §٥ج (رأس واحد مايقدرش يشترط على المصدر والوجهة مع بعض)، بذرة واحدة (٤٢)، كوربَس فصحى-أغلب والإشارة اللهجية ضعيفة.

</div>

<div dir="ltr">

# Stage 2dash — Unigram experiment: does Unigram segmentation make Arabic skip-trigrams more legible than BPE?

**Learner-scaffold notebook.** Exactly the same shape as the BPE experiment — setup, load,
decomposition, bigram, all-heads diagnostic, and heatmap cells run; the two analysis cells
(§5b and §5c) carry `# TODO(you)` for you to fill in. The only difference: this model is
trained on the **same corpus** (same chars, same seed 42, same config, vocab=12000) but with a
**Unigram** tokenizer instead of BPE — a controlled, one-variable comparison.

**Comparison hypothesis:** does Unigram segmentation make the Arabic skip-trigrams — especially
the definite article and the MSA↔Masri contrast — more *legible / morphologically aligned* than
BPE, which fragments some of those seeds?

**Honest caveat — the two train losses are NOT directly comparable:**
- Unigram: 406.5M tokens, loss_end 3.95, 73.5 min.
- BPE: 337.6M tokens, loss_end 4.96, 61.9 min.
- Unigram fragments the *same text* into ~20% MORE tokens, so the per-token distributions
  differ; **a lower Unigram loss does NOT mean a better model.** Compare which skip-trigrams
  verify, not the loss numbers.

**Upfront limits:** same as the BPE experiment — head-kind label from one diagnosis sentence,
the §5c structural bug (a head can't jointly condition on source and destination), single seed
(42), MSA-heavy corpus so the dialect signal stays weak.

</div>

<div dir="rtl">

> **الإجابة الكاملة:** إذا احتجت مساعدة أو عايز تتحقق من نتيجتك، شوف النوتبوك المرجعي:
> [`stage2_dash_skip_trigram_reference.ipynb`](stage2_dash_skip_trigram_reference.ipynb)
> — ده المفتاح الكامل مع كل السطور والنتائج.

</div>

<div dir="ltr">

> **Answer key:** if you get stuck or want to check your results, the fully worked solution
> is in [`stage2_dash_skip_trigram_reference.ipynb`](stage2_dash_skip_trigram_reference.ipynb).
> All blanked lines are there, with findings from a properly trained BPE checkpoint.

</div>

<div dir="rtl">

## ١. التفكيك بالظبط · The exact decomposition

مخرجات الموديل (logits) بتساوي بالظبط:

$$\text{logits} = \underbrace{x\,W_U}_{\text{bigram}} \;+\; \underbrace{\sum_{\text{heads, positions}} a_{\text{src}}\;(x_{\text{src}}\,W_V W_O W_U)}_{\text{skip-trigram}}$$

الموديل **من غير LayerNorm** و**shortformer** (الموضع في مسار QK بس)، فالتفكيك يطلع مظبوط بالظبط والمسار المباشر يفضل bigram حقيقي.

</div>

<div dir="ltr">

## 1. The exact decomposition

The model's logits equal, exactly, a **bigram direct path** `x·W_U` plus a sum of per-head
**skip-trigram** terms `aₛ·(xₛ·W_V·W_O·W_U)`. The model is **LayerNorm-free** with a
**shortformer** positional embedding (position in the QK path only), so the split is exact
and the direct path stays a true bigram.

</div>

<div dir="rtl">

## ٢. نحمّل الموديل والمُجزّئ · Load the model & tokenizer

نحمّل نقطة الحفظ المدرّبة (محليًا، وإلا من Hugging Face Hub) — موديل بطبقة واحدة، انتباه بس، LN-free + shortformer، مع مُجزّئ Unigram عربي (~١٢ ألف). للاختبار السريع في الـ CI نبني نسخة مصغّرة (FORCE_TINY).

</div>

In [ ]:
import os

CKPT_DIR = os.path.join(os.path.dirname(os.path.abspath(tiny.__file__)), "checkpoints", "stage2dash_unigram")
HF_REPO = ""   # local-first: the Unigram control is not pushed to the Hub; load from CKPT_DIR

# A small inline Arabic paragraph used for all eval/forward passes (no network needed,
# in-vocab for the trained model). Masri + MSA mix.
EVAL_TEXT = (
    "القطة بتاكل السمك والولد بيشرب اللبن في البيت. "
    "الجو حلو النهارده واحنا رايحين نتمشى في الشارع. "
    "المدينة كبيرة وفيها ناس كتير بتروح وتيجي كل يوم. "
    "هو قال إنه هيسافر بكرة بدري عشان الشغل المهم."
)

def _model_from_ckpt(ckpt):
    c = ckpt["config"]
    model = tiny.make_tiny_model(
        n_layers=c["n_layers"], n_heads=c["n_heads"], d_vocab=c["d_vocab"],
        n_ctx=c["n_ctx"], d_model=c["d_model"], attn_only=c["attn_only"],
        normalization_type=c["normalization_type"],
        positional_embedding_type=c["positional_embedding_type"])
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model

def _load_real():
    from tokenizers import Tokenizer
    tpath, mpath = os.path.join(CKPT_DIR, "tokenizer.json"), os.path.join(CKPT_DIR, "model.pt")
    if not (os.path.exists(tpath) and os.path.exists(mpath)):
        try:
            from huggingface_hub import hf_hub_download
            tpath = hf_hub_download(HF_REPO, "tokenizer.json")
            mpath = hf_hub_download(HF_REPO, "model.pt")
        except Exception as e:
            raise FileNotFoundError(
                f"No local checkpoint in {CKPT_DIR} and HF repo {HF_REPO!r} is unavailable "
                f"({type(e).__name__}). Either train it once with `python train_stage2dash.py` "
                f"(writes the checkpoint locally), set HF_REPO to a pushed model "
                f"(train_stage2dash.py --push-hub --hf-repo <you>/...), or set FORCE_TINY=True "
                f"for a quick structural demo on a tiny random model.") from e
    ckpt = torch.load(mpath, map_location=tiny.device(), weights_only=False)  # carries a config dict
    return _model_from_ckpt(ckpt), Tokenizer.from_file(tpath)

def _build_tiny():
    # Fast, network-free stand-in for CI: a tiny BPE on EVAL_TEXT + a tiny random model.
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFKC()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator([EVAL_TEXT] * 20,
                            trainers.BpeTrainer(vocab_size=200, special_tokens=["[UNK]"]))
    model = tiny.make_tiny_model(n_layers=1, n_heads=2, d_vocab=tok.get_vocab_size(),
        n_ctx=32, d_model=64, attn_only=True,
        normalization_type=None, positional_embedding_type="shortformer")
    model.eval()
    return model, tok

def load_model_and_tokenizer():
    return _build_tiny() if globals().get("FORCE_TINY") else _load_real()

model, tok = load_model_and_tokenizer()
VOCAB = tok.get_vocab_size()
id_to_str = {i: (tok.id_to_token(i) or "[?]") for i in range(VOCAB)}
def encode(text):
    return tok.encode(text).ids

print(f"layers=1 heads={model.cfg.n_heads} d_model={model.cfg.d_model} "
      f"n_ctx={model.cfg.n_ctx} vocab={VOCAB}")

<div dir="rtl">

## ٢ب. بذور التصنيفات · Category seeds

عشان ما نختارش الثلاثيات بإيدينا (وندّعي اللي عايزينه)، بنحدّد **مقدّمًا** أربع تصنيفات بذرة بنبحث عنها بشكل منهجي في §٥: تعابير فصحى ثابتة، صيغ دينية، أداة التعريف/الصرف، وتباين فصحى↔مصري. **النسخ الذاتي (self-copy) هو الضابط.** كل ثلاثي بيتأكّد على تمريرة محتجوزة (lift > ٠) قبل ما نعرضه — مش بنعدّ رقم ثابت، بنعدّ اللي اتأكّد فعلًا، والتصنيف الفاضي **نتيجة** برضه.

</div>

<div dir="ltr">

## 2b. Category seeds

To avoid hand-picking triples (and claiming whatever we hoped to find), we fix **four seed
categories up front** and search them systematically in §5: MSA fixed expressions, religious
formulae, the definite article / morphology, and an MSA↔Masri contrast. **Self-copy is the
control.** Every triple is checked on a held-out forward pass (lift > 0) before we show it —
we report *verified counts, not a fixed number*, and an **empty category is itself a finding**.

</div>

In [ ]:
# Category seed lexicons (Arabic-reader curated; edit freely). Self-copy is the control.
# Experiment notebooks can override SEED_LEXICONS (and CKPT_DIR above) before the §5 cells.
SEED_LEXICONS = {
    "MSA fixed expressions": ["الرغم", "بالإضافة", "حين", "بسبب", "أجل"],
    "Religious / formulaic": ["الله", "صلى", "رضي", "شاء", "عليه"],
    "Definite-article / morphology": ["ال", "الذي", "التي"],
    "MSA↔Masri contrast": ["اللي", "عايز", "دلوقتي", "الذي", "يريد", "الآن"],
}
print({k: len(v) for k, v in SEED_LEXICONS.items()})

<div dir="rtl">

## ٣. نثبت التفكيك بإيدينا · The decomposition, by hand

نفكّك الـ logits لتتابع حقيقي لجزئين — **المباشر** (تضمين × `W_U`) و**الـ skip-trigram** (مخرج الانتباه × `W_U`) — ومجموعهم لازم يطابق مخرج الموديل بالظبط: **الفرق ~صفر**.

</div>

In [ ]:
ids = torch.tensor([encode(EVAL_TEXT)[: model.cfg.n_ctx]]).to(tiny.device())
logits, cache = model.run_with_cache(ids)

direct = cache["resid_pre", 0] @ model.W_U + model.b_U   # bigram path       <- (1, ctx, V)
skip   = cache["attn_out", 0] @ model.W_U                # skip-trigram path  <- (1, ctx, V)
print("max |(direct + skip) - logits|:", float((direct + skip - logits).abs().max()))  # ~0

# how big is the skip-trigram contribution relative to the bigram skeleton?
print("||skip|| / ||direct|| (logit space):", round(float(skip.norm() / direct.norm()), 2))

<div dir="rtl">

## ٤. دائرة الـ bigram (المسار المباشر) · The bigram circuit

`W_E W_U` هي **هيكل الـ bigram**: لكل توكن، إيه التوكنز اللي بيرجّحها مباشرة من غير سياق. التوكنز الفصحى المتكررة بتطلع **متلازمات نظيفة** (على الرغم، من أجل، البيت الأبيض). التوكنز المصري (زي "اللي") **ضعيفة** لأن الكوربَس فصحى-أغلب — وده بنفسه ملاحظة عن ندرة بيانات اللهجة.

</div>

<div dir="ltr">

## 4. The bigram circuit

`W_E W_U` is the **bigram skeleton**: for each token, what it directly favours next, with no
context. Frequent MSA tokens give **clean collocations**; Masri tokens (e.g. "اللي") are
**weak** because the corpus is MSA-heavy — itself an observation about dialect-data scarcity.

</div>

In [ ]:
bigram = (model.W_E @ model.W_U).detach().cpu()   # (V, V): current token -> next token

def bigram_top(word, k=6):
    seq = encode(word)
    if not seq:
        return f"{word!r}: empty"
    tid = seq[-1]
    _, idx = torch.topk(bigram[tid], k)
    return f"{id_to_str[tid]:>8} ->  " + " · ".join(id_to_str.get(int(j), "[?]") for j in idx)

print("MSA (well-trained) — crisp collocations:")
for w in ["في", "من", "على", "كان", "البيت"]:
    print(" ", bigram_top(w))
print("\nMasri (data-starved here; corpus is MSA-heavy) — weaker / fragmented:")
for w in ["اللي", "عايز", "دلوقتي"]:
    print(" ", bigram_top(w))

<div dir="rtl">

## ٥. دوائر الـ skip-trigram — الطريقة الأمينة · The honest method

كل واحد من الرؤوس = جدول skip-trigram من دائرتين:
- **QK** = `W_E W_Q W_K^T W_E^T`: لتوكن وجهة، **هنبص على مين** (الـ source)؟
- **OV** = `W_E W_V W_O W_U`: الـ source ده، **إيه الناتج اللي بيرجّحه**؟

الطريقة الأمينة من ٣ خطوات: **(١) تشخيص** — أي رأس بيبص بالمحتوى أصلًا (رأس موضعي مايقدرش يستضيف skip-trigram محتوى)؛ **(٢) تجميع مرشّحين** من بذور التصنيفات + بحث غير موجَّه، مرتّبين بـ `OV·QK`؛ **(٣) تأكيد محتجوز**: نبني `[source … source … dest]` ونشوف هل الموديل الكامل بيرفع `P(output)` فوق خط الـ bigram (lift > ٠). بنعرض **اللي اتأكّد بس**.

**حدود مقدّمًا:** التصنيف بيتعمل على **جملة تشخيص واحدة**؛ وفي تجربة BPE بعض البذور اتكسّرت (عايز→عا) فطلعت تصنيفات ضعيفة — هل Unigram بيحافظ عليها كاملة ويخلّي التصنيفات دي أوضح؟ ده اللي بتختبره. وفي **عيب بنيوي** (§٥ج): رأس واحد مايقدرش يشترط على الـ source والـ dest مع بعض.

</div>

<div dir="ltr">

## 5. The skip-trigram circuits — the honest method

Each head is a skip-trigram table from two circuits: **QK** (`W_E W_Q W_Kᵀ W_Eᵀ`,
which source to attend to) and **OV** (`W_E W_V W_O W_U`, what the attended source promotes).

The honest pipeline is three steps: **(1) diagnose** which heads attend by *content* at all (a
positional head — BOS or previous-token — can't host a content skip-trigram); **(2) pool
candidates** from the seed categories *and* an unsupervised scan, ranked by `OV·QK`; **(3)
verify held-out**: build `[source … source … dest]` and check the full model raises
`P(output)` above the bigram baseline (lift > 0). We print **only verified triples**.

**Limits up front:** the head-kind label is from **one diagnosis sentence**. BPE fragmented the
seeds (عايز→عا) so some categories came back weak — whether Unigram keeps them whole and makes
those categories more legible is exactly what you are testing. And a **structural bug**
(§5c) bounds it: a single head cannot jointly condition on *both* source and destination.

</div>

In [ ]:
import skip_trigrams as st

# §5a — diagnose each head on ONE sentence. (bos/prev are the raw masses the classifier
# thresholds at 0.5; printing them keeps borderline heads honest.)
sent = "هو قال إنه هيسافر بكرة بدري عشان الشغل المهم"
sids = encode(sent)[: model.cfg.n_ctx]
_, hcache = model.run_with_cache(torch.tensor([sids]).to(tiny.device()))
print("Per-head attention kind (positional heads can't host content skip-trigrams):")
for h in range(model.cfg.n_heads):
    patt = hcache["pattern", 0][0, h].detach().cpu()
    n = patt.shape[0]
    den = max(n - 1, 1)
    bos = sum(float(patt[i, 0]) for i in range(1, n)) / den
    prev = sum(float(patt[i, i - 1]) for i in range(1, n)) / den
    print(f"  head {h}: {st.head_attention_kind(patt):>10}   (bos={bos:.2f}  prev={prev:.2f})")

In [ ]:
import skip_trigrams as st

# §5b — pool ~100 candidates PER CATEGORY, verify the WHOLE pool held-out, keep survivors.
# Honest catch (you'll see it in §7): per_source_dests=4 pairs each source->output OV-promotion
# with up to 4 destinations, most of them spurious high-QK *fragments*. Since OV[source] is
# destination-independent (§5c), the same promotion verifies under several dests — so the verified
# COUNT inflates the number of distinct findings. Track distinct source->output too, and feature
# only triples whose destination is a real co-occurring token.
# Helpers: st.candidate_pool(..., per_source_outputs=5, per_source_dests=4) | st.dedup_triples |
#          st.verify_triple | st.top_per_group | st.triple_table_html
FREQ = min(2500, VOCAB)
CONTENT_HEADS = [h for h in range(model.cfg.n_heads)
                 if st.head_attention_kind(
                     hcache["pattern", 0][0, h].detach().cpu()) == "content"] or [0]

def pool_for(seed_words, heads, want=100):
    src = set(st.seed_ids(encode, id_to_str, seed_words, FREQ)) if seed_words else None
    per_out, per_dst = (5, 4) if seed_words else (1, 1)   # seeds expand; unsupervised scans broad
    cands = []
    for h in heads:
        # TODO(you): pool this head's candidates and tag the head onto each:
        #   for v in st.candidate_pool(model, head=h, freq=FREQ, top_n=want, sources=src,
        #                              per_source_outputs=per_out, per_source_dests=per_dst):
        #       cands.append({**v, "head": h})
        pass
    cands = st.dedup_triples(cands)                       # same triple from two heads -> one
    cands.sort(key=lambda c: c.get("score", 0), reverse=True)
    cands = cands[:want]                                 # the ~100 we pick from
    # TODO(you): verify EVERY candidate (not a top slice) so "most representative" is best-of-100:
    #   verified = [st.verify_triple(model, c) for c in cands]
    verified = []  # TODO(you): replace [] with the list comprehension above
    verified.sort(key=lambda v: v.get("lift", 0), reverse=True)
    return verified

def show(label, verified, k=6):
    real = [v for v in verified if v.get("verified")]
    strong = [v for v in real if v.get("lift", 0) >= 0.05]
    print(f"\n{label}: {len(real)} verified ({len(strong)} with lift>=0.05) of {len(verified)} checked")
    for v in real[:k]:
        s, d, o = id_to_str.get(v['source']), id_to_str.get(v['dest']), id_to_str.get(v['output'])
        print(f"   h{v['head']} [{s} ... {d} -> {o}]   lift={v['lift']:+.3f}  score={v['score']:.1f}")

ALL_VERIFIED = []
for label, seeds in SEED_LEXICONS.items():
    v = pool_for(seeds, CONTENT_HEADS)
    for r in v:
        r["group"] = label
    show(label, v)
    ALL_VERIFIED += [r for r in v if r.get("verified")]

# Self-copy control — what a copy-dominated head looks like (blanked for you to fill):
show("Self-copy baseline (control . head %d)" % CONTENT_HEADS[0], [])
# TODO(you): replace [] above with:
#   [{**v, "head": CONTENT_HEADS[0]} for v in st.verify_pool(
#       model, st.candidate_pool(model, head=CONTENT_HEADS[0], freq=FREQ,
#                                include_self_copy=True, top_n=20))]

unsup = pool_for(None, CONTENT_HEADS)
for r in unsup:
    r["group"] = "Unsupervised"
show("Unsupervised - what else is in here?", unsup)
ALL_VERIFIED += [r for r in unsup if r.get("verified")]

# Distinct source->output vs raw verified count: the gap is dest-variant inflation (the §5c bug).
print("\nDistinct source->output per category (verified count inflates findings via dest-variants):")
for label in list(SEED_LEXICONS) + ["Unsupervised"]:
    vs = [v for v in ALL_VERIFIED if v.get("group") == label]
    nd = len({(v["source"], v["output"]) for v in vs})
    print(f"   {label}: {len(vs)} verified, {nd} distinct source->output")

# A triple is a GENUINE 3-way frame only if its DESTINATION is a real co-occurring token (not a
# spurious QK fragment). Curate by hand — gloss keyed on the FULL (source, dest, output):
GLOSS = {
    # ("source_tok", "dest_tok", "output_tok"): "what it encodes",   # TODO(you): add your finds
}
def glossed(v):
    return GLOSS.get((id_to_str.get(v["source"]), id_to_str.get(v["dest"]), id_to_str.get(v["output"])))

# Featured = genuine frames. A category with no glossable triple is null *as a skip-trigram*
# (only the §5c dest-independent OV promotion fired), even if many triples technically verified.
featured, NULLS = [], []
for label in list(SEED_LEXICONS) + ["Unsupervised"]:
    vs = [v for v in ALL_VERIFIED if v.get("group") == label]
    g = [v for v in vs if glossed(v)]
    if g:
        featured.append(max(g, key=lambda v: v["lift"]))
    else:
        NULLS.append(label)
BOARD = sorted(ALL_VERIFIED, key=lambda v: v.get("lift", 0), reverse=True)[:12]
# Evidence for the split: the destinations each (source->output) actually verified with.
# A genuine frame's set contains a real co-occurring token (من, ي); a dest-independent OV
# promotion's set is only spurious QK fragments (حد/حض/فض/وض) — the §5c bug, made visible.
def _dests(source, output):
    ds = [id_to_str.get(v["dest"]) for v in ALL_VERIFIED
          if v["source"] == source and v["output"] == output]
    return list(dict.fromkeys(ds))  # order-preserving unique (a source may seed two categories)

print("\nDestination evidence (a genuine frame rides a real token; a bug rides a fragment bag):")
for v in featured:
    print(f"   FRAME ({id_to_str.get(v['source'])} -> {id_to_str.get(v['output'])}): "
          f"dests = {_dests(v['source'], v['output'])}")
for label in NULLS:
    seen, picks = set(), []
    for v in sorted((x for x in ALL_VERIFIED if x.get("group") == label),
                    key=lambda x: x["lift"], reverse=True):
        k = (v["source"], v["output"])
        if k not in seen:
            seen.add(k)
            picks.append(v)
        if len(picks) == 2:
            break
    for v in picks:
        print(f"   BUG   ({id_to_str.get(v['source'])} -> {id_to_str.get(v['output'])}): "
              f"dests = {_dests(v['source'], v['output'])}  [{label}]")

print(f"\ngenuine frames: {len(featured)}; no-frame categories (OV-promotion only): {NULLS or 'none'}")

<div dir="rtl">

### ٥ج. العيب البنيوي · The structural bug

الثلاثيات اللي اتأكّدت فوق **حقيقية** — بس كل واحد منها هو **أحسن شريحة** للرأس، مش اشتراط ثلاثي كامل. السبب: الـ `OV[source]` **مستقل عن الوجهة** — الوجهة بتقرر *هل* هتبص على الـ source، مش *إيه* اللي هيترفع. يبقى أي source قوي بيرفع **أكتر من ناتج** مهما كانت الوجهة. ده العيب الكلاسيكي بتاع الـ skip-trigram في *Elhage et al. (2021)*: `keep…in→mind` بيجرّ معاه `keep…in→bay`.

</div>

<div dir="ltr">

### 5c. The structural bug

The verified triples above are **real** — but each is the head's **best slice**, not full
three-way conditioning. Reason: `OV[source]` is **destination-independent** — the destination
decides *whether* you attend to the source, not *what* gets promoted. So a strong source
raises **several outputs** regardless of destination. This is the classic skip-trigram bug of
*Elhage et al. (2021)*: wanting `keep…in→mind` forces `keep…in→bay` along with it.

</div>

In [ ]:
# §5c — show the bug as a table: one source promotes a SECOND output regardless of destination.
# OV[source] is destination-independent, so wanting [.. -> output] drags [.. -> other] with it.
from IPython.display import HTML, display

ref = max((v for v in featured if glossed(v)), key=lambda v: v["lift"], default=None)
if ref is not None:
    _, OV = st.head_circuits(model, ref["head"])
    s = ref["source"]
    # TODO(you): the rides-along output = strongest OV output that isn't the source or the wanted one:
    #   extra = [int(i) for i in torch.topk(OV[s, :FREQ], 4).indices
    #            if int(i) not in (s, ref["output"])]
    #   o_bug = extra[0] if extra else ref["output"]
    o_bug = ref["output"]  # TODO(you): replace with the rides-along output (see hint above)
    bug_rows = [
        {"source": s, "dest": ref["dest"], "output": ref["output"], "head": ref["head"],
         "group": "wanted", "gloss": glossed(ref)},
        {"source": s, "dest": ref["dest"], "output": o_bug, "head": ref["head"],
         "group": "rides along", "gloss": "same OV[source] promotes it too"},
    ]
    display(HTML(st.triple_table_html(
        bug_rows, id_to_str,
        title="the structural skip-trigram bug - one source, two outputs",
        note="the destination only gates WHETHER attention reaches the source, never WHICH output "
             "rises (keep...in->mind forces keep...in->bay)")))
else:
    print("fill in §5b first - no glossable verified triple yet to show the bug on.")

<div dir="rtl">

## ٦. خريطة الانتباه · Attention heatmap

دائرة الـ QK بتقرر نمط الانتباه. نشوفه لرأس واحد على جملة حقيقية: كل صف (وجهة) بيوزّع انتباهه على التوكنز اللي قبله. (بنعرض من اليمين للشمال زي العربي.)

</div>

In [ ]:
sent = "هو قال إنه هيسافر بكرة بدري عشان الشغل"
sids = encode(sent)[: model.cfg.n_ctx]
str_toks = [id_to_str.get(i, "[?]") for i in sids]
_, hcache = model.run_with_cache(torch.tensor([sids]).to(tiny.device()))
pattern = hcache["pattern", 0][0, 0].detach().cpu().numpy()  # head 0  <- (seq, seq)

fig = go.Figure(go.Heatmap(z=pattern, x=str_toks, y=str_toks, colorscale="Blues"))
fig.update_layout(title="Head 0 attention — " + sent, xaxis=dict(side="top"), height=460, width=560)
fig.update_xaxes(autorange="reversed")  # RTL
fig.update_yaxes(autorange="reversed")
fig.show()

<div dir="rtl">

## ٧. معبّر بشكل مفاجئ · Surprisingly expressive

دي الخلاصة (punchline). بعد ما تملا §٥ب، الكود تحت بيعرض **الأكثر تمثيلًا اللي اتأكّد** لكل تصنيف على شكل جدول زي ورقة *Elhage et al.*: `[source … dest] → output` مع الرفع (lift) والرأس والتفسير. التفسير ده — من قاموس `GLOSS` اللي بتكتبه إنت — هو اللي يخلّيه «معبّر»: كل سطر قالب عربي حقيقي اتعلمه الموديل، مش رقم مجمّع. الجدول التاني = كل اللي اتأكّد (أعلى ١٢ بالرفع)، والرقم المجمّع اتنقل لحاشية.

</div>

<div dir="ltr">

## 7. Surprisingly expressive

This is the punchline. Once you fill in §5b, the code below renders the **most representative
verified** triple per category as an *Elhage et al.*-style table — `[source … dest] → output`
with its lift, the head it lives on, and a gloss. That gloss — from the `GLOSS` dict you write —
is what makes it *expressive*: every row is a real template the model learned, not an aggregate.
The second table is the full verified board (top 12 by lift); the aggregate number is a footnote.

</div>

In [ ]:
# §7 — the punchline as paper-style tables: the genuine 3-way frames (meaningful destination),
# then the raw top-12 board. The aggregate context-shift is demoted to a footnote.
from IPython.display import HTML, display

def with_gloss(v):
    return {**v, "gloss": glossed(v) or ""}

# aggregate (footnote, not headline): how far does the skip-trigram move the bigram distribution?
W = min(16, model.cfg.n_ctx)
flat = encode(EVAL_TEXT)
wins = [flat[i:i + W] for i in range(0, len(flat) - W, 4)][:32]
batch = torch.tensor(wins).to(tiny.device())
with torch.no_grad():
    logits2, cache2 = model.run_with_cache(batch)
    direct2 = cache2["resid_pre", 0] @ model.W_U + model.b_U
    p_full, p_dir = torch.softmax(logits2, -1), torch.softmax(direct2, -1)
reshape = float((p_full - p_dir).abs().max(-1).values.mean())
changed = float((p_full.argmax(-1) != p_dir.argmax(-1)).float().mean())

null_note = ("no genuine frame — dest-independent OV promotions only (§5c): " + ", ".join(NULLS)) \
    if NULLS else "every category produced a genuine frame"
display(HTML(st.triple_table_html(
    [with_gloss(v) for v in featured], id_to_str,
    title="أطر حقيقية (وجهة ذات معنى) · genuine 3-way frames (meaningful destination)",
    note="genuine frames only, after dropping dest-independent promotions.  " + null_note)))
display(HTML(st.triple_table_html(
    [with_gloss(v) for v in BOARD], id_to_str,
    title="أعلى ١٢ بالرفع (خام) · raw top-12 by lift (unfiltered)",
    note=f"context reshapes the next-token dist by avg {reshape:.3f} and flips top-1 at "
         f"{changed * 100:.0f}% of positions")))

<div dir="rtl">

## ٨. الخلاصة · Recap — مقارنة (اكتبها بنفسك بعد ما تكمّل التجربة)

بعد ما تكمّل خلايا الـ TODO ↑، دوّن مقارنتك مع تجربة BPE هنا:

- **التفكيك:** هل الفرق بين (direct + skip) والـ logits ≈ صفر؟
- **التشخيص:** كام رأس طلع content في موديل Unigram؟
- **التصنيفات (المقارنة):** أي تصنيفات عربية اتأكّدت على Unigram؟ قارن بـ BPE — أداة التعريف والتباين فصحى↔مصري بقت أوضح ولا فضلت ضعيفة؟
- **التجزئة:** بُص على تجزئة البذور (ال، عايز، دلوقتي) في المُجزّئين — مين بيحافظ على الكلمة كاملة؟ ده بيفسّر فرق المحاذاة الصرفية؟
- **القراءة الأمينة:** افتكر إن الخسارتين مش قابلتين للمقارنة (Unigram توكنز أكتر بـ~٢٠٪) — استنتاجك لازم يكون على الـ skip-trigrams اللي اتأكّدت، مش على رقم الخسارة.
- **اللقطة (§٧):** اكتب مدخلات `GLOSS` بتاعتك — مين أكثر ثلاثي تمثيلًا لكل تصنيف في الجدول؟

</div>

<div dir="ltr">

## 8. Recap — fill in after you complete the experiment (a comparison)

After filling in the TODO cells above, record your **BPE-vs-Unigram** comparison here:

- **Decomposition:** does max|(direct + skip) - logits| ≈ 0?
- **Diagnosis:** how many heads are "content" in the Unigram model?
- **Categories (the comparison):** which Arabic seed categories verified on Unigram? Compared
  with BPE, did the **definite-article/morphology** and **MSA↔Masri** categories become more
  legible, or stay weak?
- **Segmentation:** inspect how each tokenizer splits the seeds (ال, عايز, دلوقتي) — which keeps
  the word whole? Does that explain any difference in morphological alignment?
- **Honest read:** remember the two losses are NOT comparable (Unigram has ~20% more tokens) —
  base your conclusion on which skip-trigrams verified, not on the loss number.
- **The punchline (§7):** write your `GLOSS` entries — which triple is the most representative per category in the table?

</div>

<div dir="ltr">

*(The fully worked BPE answer key is
[`stage2_dash_skip_trigram_reference.ipynb`](stage2_dash_skip_trigram_reference.ipynb); this
Unigram notebook is yours to fill in and compare against it.)*

</div>